### ลำดับขั้นตอน
1. **Setup & Config** — ตั้งค่า API, path, environment variables
2. **Load Knowledge Base** — โหลดไฟล์ .md ทั้งหมด
3. **Chunking & Indexing** — แบ่งข้อความ + สร้าง BM25 + TF-IDF index  
4. **Hybrid Retrieval** — ค้นหาด้วย BM25 + TF-IDF + Keyword แบบ weighted
5. **Prompt Engineering** — ออกแบบ prompt ภาษาไทยพร้อม few-shot
6. **LLM Inference** — เรียก Thai LLM API พร้อม retry อัตโนมัติ
7. **Parallel Pipeline** — ประมวลผลคำถามแบบ parallel + retry + fallback
8. **Checkpoint/Resume** — บันทึก/โหลด checkpoint ระหว่างทาง
9. **Multi-Model Ensemble** — รวมหลาย model ด้วย majority voting
10. **Generate Submission** — สร้าง submission.csv พร้อม validate
11. **Analysis & Debug** — วิเคราะห์ผลลัพธ์และ debug รายคำถาม


## ขั้นตอนที่ 1: Setup & Configuration

In [ ]:
# ติดตั้ง library ที่จำเป็น (BM25 และ sklearn สำหรับ Hybrid Retrieval)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'rank_bm25', 'scikit-learn', 'pandas', 'tqdm'])


0

In [ ]:
import os, re, json, time, csv, requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

# ตั้งค่า API
API_KEY = os.environ.get("THAI_LLM_API_KEY", "zEJ2JyfKmDAguvcUgVndXkNGF7g19vsf")

MODEL_ENDPOINTS = {
    "openthaigpt": "http://thaillm.or.th/api/openthaigpt/v1",
    "pathumma":    "http://thaillm.or.th/api/pathumma/v1",
    "typhoon":     "http://thaillm.or.th/api/typhoon/v1",
    "kbtg":        "http://thaillm.or.th/api/kbtg/v1",
}
MODEL_NAMES = {
    "openthaigpt": "OpenThaiGPT-ThaiLLM-8B-instruct-v7.2",
    "pathumma":    "Pathumma-ThaiLLM-qwen3-8b-think-3.0.0",
    "typhoon":     "Typhoon-S-ThaiLLM-8B-Instruct",
    "kbtg":        "THaLLE-0.2-ThaiLLM-8b-fa",
}

# ตั้งค่าหลักจาก environment variable (สไตล์ main.py เดิม)
PRIMARY_MODEL  = os.environ.get("PRIMARY_MODEL", "openthaigpt")
SUBMISSION_CSV = Path(os.environ.get("SUBMISSION_CSV", "submission.csv"))
MAX_WORKERS    = max(1, int(os.environ.get("MAX_WORKERS", "3")))          # จำนวน thread parallel
RERUN_FAILED   = max(0, int(os.environ.get("RERUN_FAILED_PASSES", "2"))) # จำนวนรอบ retry

# ค่า default ของ LLM request
DEFAULT_MAX_TOKENS  = 512
DEFAULT_TEMPERATURE = 0.1
REQUEST_TIMEOUT     = 60  # วินาที

# Path ของคุณ — ใช้ Path() ครอบ os.path.join เพื่อให้ได้ Path object เสมอ
BASE_DIR        = Path("/content/drive/MyDrive/test")
KB_DIR          = Path(os.path.join(BASE_DIR, "knowledge_base"))
QUESTIONS_FILE  = Path(os.path.join(BASE_DIR, "questions.csv"))
CHECKPOINT_FILE = Path("rag_checkpoint.json")

print("✅ โหลด configuration สำเร็จ")
print(f"   Primary model : {MODEL_NAMES[PRIMARY_MODEL]}")
print(f"   MAX_WORKERS   : {MAX_WORKERS}")
print(f"   RERUN_FAILED  : {RERUN_FAILED}")
print(f"   Submission    : {SUBMISSION_CSV}")
print(f"   KB_DIR        : {KB_DIR}")


✅ โหลด configuration สำเร็จ
   Primary model : OpenThaiGPT-ThaiLLM-8B-instruct-v7.2
   MAX_WORKERS   : 3
   RERUN_FAILED  : 2
   Submission    : submission.csv
   KB_DIR        : /content/drive/MyDrive/test/knowledge_base


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ขั้นตอนที่ 2: Load Knowledge Base

In [ ]:
def load_knowledge_base(kb_dir: Path) -> list[dict]:
    """โหลดไฟล์ .md ทั้งหมดจาก knowledge base directory"""
    documents = []
    md_files  = sorted(kb_dir.rglob("*.md"))

    for fp in md_files:
        try:
            content = fp.read_text(encoding="utf-8").strip()
        except Exception as e:
            print(f"[WARN] อ่านไฟล์ {fp} ไม่ได้: {e}")
            continue

        # หมวดหมู่ = ชื่อโฟลเดอร์ย่อยชั้นแรก
        try:
            category = fp.relative_to(kb_dir).parts[0] if len(fp.relative_to(kb_dir).parts) > 1 else "root"
        except ValueError:
            category = "root"

        documents.append({
            "path":     str(fp),
            "category": category,
            "filename": fp.name,
            "content":  content,
        })

    return documents


# โหลด knowledge base
documents = load_knowledge_base(KB_DIR)
print(f" โหลด {len(documents)} เอกสารจาก knowledge base")

# แสดงสัดส่วนแต่ละหมวด
cat_counts = Counter(d["category"] for d in documents)
for cat, cnt in sorted(cat_counts.items()):
    print(f"   {cat}: {cnt} ไฟล์")

# โหลดคำถาม
questions_df = pd.read_csv(QUESTIONS_FILE)
print(f"\n โหลด {len(questions_df)} คำถาม")
print(f"   Columns: {list(questions_df.columns)}")
display(questions_df.head(3))


 โหลด 118 เอกสารจาก knowledge base
   policies: 5 ไฟล์
   products: 110 ไฟล์
   store_info: 3 ไฟล์

 โหลด 100 คำถาม
   Columns: ['id', 'question', 'choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10']


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


## ขั้นตอนที่ 3: Text Chunking & Indexing (BM25 + TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi


def simple_thai_tokenize(text: str) -> list[str]:
    """Tokenizer ง่าย — แยกด้วย whitespace และกรองอักขระพิเศษออก"""
    text   = re.sub(r"[\r\n\t]+", " ", text)
    text   = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9\s]", " ", text)
    tokens = text.lower().split()
    return [t for t in tokens if len(t) >= 2]  # กรอง token สั้นเกินออก


def chunk_documents(documents: list[dict], chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """แบ่งเอกสารเป็น chunk ขนาดไม่เกิน chunk_size ตัวอักษร
    มี overlap เพื่อรักษา context ที่ขอบ chunk"""
    chunks = []
    for doc in documents:
        content = doc["content"]
        if len(content) <= chunk_size:
            # เอกสารสั้นพอ — ไม่ต้องแบ่ง
            chunks.append({**doc, "chunk_id": f"{doc['filename']}_0", "chunk_text": content})
        else:
            start, idx = 0, 0
            while start < len(content):
                end = min(start + chunk_size, len(content))
                chunks.append({**doc, "chunk_id": f"{doc['filename']}_{idx}", "chunk_text": content[start:end]})
                idx   += 1
                start += chunk_size - overlap  # เลื่อน window พร้อม overlap
    return chunks


# สร้าง chunks
chunks       = chunk_documents(documents, chunk_size=500, overlap=50)
chunk_texts  = [c["chunk_text"] for c in chunks]
print(f"🔪 แบ่งเอกสารได้ทั้งหมด {len(chunks)} chunks")

# สร้าง BM25 index (Okapi BM25 — ดีกว่า TF-IDF สำหรับภาษาไทย)
tokenized_chunks = [simple_thai_tokenize(t) for t in chunk_texts]
bm25_index       = BM25Okapi(tokenized_chunks)
print("✅ BM25 index พร้อมใช้งาน")

# สร้าง TF-IDF index (ใช้ร่วมกับ BM25 แบบ hybrid)
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=simple_thai_tokenize,
    token_pattern=None,
    min_df=1,
    max_features=50000,
    sublinear_tf=True,   # log-scaling ช่วยลด bias คำที่ปรากฏบ่อยมาก
)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)
print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")


🔪 แบ่งเอกสารได้ทั้งหมด 907 chunks
✅ BM25 index พร้อมใช้งาน
✅ TF-IDF matrix shape: (907, 8498)


## ขั้นตอนที่ 4: Hybrid Retrieval (BM25 + TF-IDF + Keyword)

In [ ]:
def retrieve_bm25(query: str, top_k: int = 5) -> list[dict]:
    """ค้นหาด้วย BM25 — ดีกับคำที่ตรงตัว"""
    tokens  = simple_thai_tokenize(query)
    scores  = bm25_index.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], "score": float(scores[i]), "method": "bm25"}
            for i in top_idx if scores[i] > 0]


def retrieve_tfidf(query: str, top_k: int = 5) -> list[dict]:
    """ค้นหาด้วย TF-IDF cosine similarity — ดีกับ semantic similarity"""
    q_vec   = tfidf_vectorizer.transform([query])
    sims    = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [{**chunks[i], "score": float(sims[i]), "method": "tfidf"}
            for i in top_idx if sims[i] > 0]


def retrieve_keyword(query: str, top_k: int = 5) -> list[dict]:
    """ค้นหาด้วย exact keyword match — จับคำเฉพาะได้ดี"""
    tokens   = simple_thai_tokenize(query)
    keywords = [t for t in tokens if len(t) >= 3] or tokens  # เอาคำยาวกว่า 3 ตัว
    results  = []
    for chunk in chunks:
        text_lower  = chunk["chunk_text"].lower()
        match_count = sum(1 for kw in keywords if kw in text_lower)
        if match_count > 0:
            results.append({**chunk, "score": match_count / len(keywords), "method": "keyword"})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


def hybrid_retrieve(query: str, top_k: int = 6) -> list[dict]:
    """รวม BM25 + TF-IDF + Keyword ด้วย weighted score
    น้ำหนัก: BM25=0.50, TF-IDF=0.35, Keyword=0.15"""
    bm25_r  = retrieve_bm25(query, top_k=top_k)
    tfidf_r = retrieve_tfidf(query, top_k=top_k)
    kw_r    = retrieve_keyword(query, top_k=top_k)

    # รวม score แบบ normalize แล้ว weighted sum
    scored: dict[str, dict] = {}

    def _add(results, weight):
        if not results:
            return
        max_s = max(r["score"] for r in results) or 1.0
        for r in results:
            cid = r["chunk_id"]
            norm = r["score"] / max_s  # normalize เป็น [0,1]
            if cid not in scored:
                scored[cid] = {**r, "hybrid_score": 0.0}
            scored[cid]["hybrid_score"] += weight * norm

    _add(bm25_r,  0.50)   # BM25 สำคัญที่สุด
    _add(tfidf_r, 0.35)   # TF-IDF รองลงมา
    _add(kw_r,    0.15)   # Keyword bonus

    ranked = sorted(scored.values(), key=lambda x: x["hybrid_score"], reverse=True)
    return ranked[:top_k]


# ทดสอบ retrieval
test_q = "ราคาสินค้า"
test_r = hybrid_retrieve(test_q, top_k=3)
print(f"🔍 ทดสอบ query: '{test_q}'")
for r in test_r:
    print(f"   [{r['category']}] {r['filename']} score={r['hybrid_score']:.3f}")


🔍 ทดสอบ query: 'ราคาสินค้า'
   [policies] warranty_policy.md score=0.150
   [products] DN-LT-010_daonuea_stormbook_g9_titan.md score=0.150
   [products] DN-LT-014_daonuea_creatorbook_16_oled.md score=0.150


## ขั้นตอนที่ 5: Prompt Engineering

In [ ]:
# System prompt — สั่ง model ให้ตอบแค่ตัวเลข 1-10
SYSTEM_PROMPT = """คุณเป็น AI ผู้ช่วยที่เชี่ยวชาญในการตอบคำถามเกี่ยวกับร้านฝ้ายไหม (FahMai) โดยอ้างอิงจากข้อมูลในฐานข้อมูลที่ให้มาเท่านั้น

กฎสำคัญ:
1. อ่านข้อมูลที่ให้มาอย่างละเอียด
2. ตอบโดยอ้างอิงจากข้อมูลที่ให้มาเท่านั้น ห้ามใช้ความรู้ภายนอก
3. หากไม่มีข้อมูลในฐานข้อมูล ให้ตอบด้วยเลข 9
4. หากคำถามไม่เกี่ยวข้องกับฐานข้อมูลสินค้าร้านฝ้ายไหม ให้ตอบด้วยเลข 10
5. ตอบด้วยตัวเลขเพียงอย่างเดียว (1-10) ห้ามอธิบายเพิ่มเติม"""

# Few-shot examples — สอน model ก่อนให้ตอบถูกรูปแบบ
FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """ข้อมูลจากฐานข้อมูล:
ผ้าไหมมัดหมี่ ราคา 1,500 บาท/ผืน มีสีแดง สีน้ำเงิน สีเขียว

คำถาม: ผ้าไหมมัดหมี่มีสีอะไรบ้าง?
ตัวเลือก:
1. สีแดง สีน้ำเงิน
2. สีแดง สีน้ำเงิน สีเขียว
3. สีเหลือง สีส้ม
4. สีขาว สีดำ
5. สีม่วง สีชมพู
6. สีเทา สีน้ำตาล
7. สีแดง สีเขียว สีเหลือง
8. สีน้ำเงิน สีม่วง สีเขียว
9. ไม่มีข้อมูลนี้ในฐานข้อมูล
10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

ตอบด้วยตัวเลขเพียงอย่างเดียว:""",
    },
    {"role": "assistant", "content": "2"},
    {
        "role": "user",
        "content": """ข้อมูลจากฐานข้อมูล:
นโยบายการคืนสินค้า: สามารถคืนสินค้าได้ภายใน 7 วัน หากสินค้ามีตำหนิจากโรงงาน

คำถาม: ราคาหุ้นของบริษัท Apple วันนี้เป็นเท่าไร?
ตัวเลือก:
1. 150 ดอลลาร์
2. 175 ดอลลาร์
3. 200 ดอลลาร์
4. 120 ดอลลาร์
5. 160 ดอลลาร์
6. 180 ดอลลาร์
7. 190 ดอลลาร์
8. 145 ดอลลาร์
9. ไม่มีข้อมูลนี้ในฐานข้อมูล
10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

ตอบด้วยตัวเลขเพียงอย่างเดียว:""",
    },
    {"role": "assistant", "content": "10"},
]


def build_rag_prompt(question: str, choices: list[str], context_chunks: list[dict]) -> list[dict]:
    """สร้าง message list สำหรับส่ง LLM พร้อม context จาก retrieval"""
    # สร้าง context string จาก retrieved chunks
    if context_chunks:
        context_parts = [
            f"[ข้อมูลที่ {i}] (ไฟล์: {c['filename']})\n{c['chunk_text']}"
            for i, c in enumerate(context_chunks, 1)
        ]
        context_str = "\n\n".join(context_parts)
    else:
        context_str = "ไม่พบข้อมูลที่เกี่ยวข้องในฐานข้อมูล"

    # สร้าง choices string
    choices_str = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))

    user_message = f"""ข้อมูลจากฐานข้อมูล:
{context_str}

คำถาม: {question}
ตัวเลือก:
{choices_str}

ตอบด้วยตัวเลขเพียงอย่างเดียว (1-10):"""

    # รวม system + few-shot + user message
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {"role": "user", "content": user_message},
    ]


print("✅ Prompt functions พร้อมใช้งาน")
print(f"   System prompt: {len(SYSTEM_PROMPT)} ตัวอักษร")
print(f"   Few-shot: {len(FEW_SHOT_EXAMPLES)//2} คู่")


✅ Prompt functions พร้อมใช้งาน
   System prompt: 401 ตัวอักษร
   Few-shot: 2 คู่


## ขั้นตอนที่ 6: LLM Inference Functions

In [ ]:
def strip_think_tags(text: str) -> str:
    """ลบ <think>...</think> block ที่ model บางตัว (เช่น Pathumma/qwen3-think) สร้างขึ้น"""
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    return text.strip()


def call_llm(
    messages:    list[dict],
    model_key:   str   = PRIMARY_MODEL,
    max_tokens:  int   = DEFAULT_MAX_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    retries:     int   = 3,
    retry_delay: float = 5.0,
) -> str:
    """เรียก Thai LLM API ด้วย requests
    - ใช้ header 'apikey' (ไม่ใช่ Authorization: Bearer)
    - model body ให้ใส่ '/model' เสมอ
    - retry อัตโนมัติเมื่อ timeout หรือ HTTP error
    """
    url     = f"{MODEL_ENDPOINTS[model_key]}/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": API_KEY}
    payload = {
        "model":       "/model",
        "messages":    messages,
        "max_tokens":  max_tokens,
        "temperature": temperature,
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            raw_text = resp.json()["choices"][0]["message"]["content"]
            return strip_think_tags(raw_text)  # ลบ <think> tags ก่อน return
        except requests.exceptions.Timeout:
            print(f"[WARN] Timeout attempt {attempt}/{retries} ({model_key})")
        except requests.exceptions.HTTPError as e:
            print(f"[WARN] HTTP {e.response.status_code} attempt {attempt}/{retries}: {e}")
        except Exception as e:
            print(f"[WARN] Error attempt {attempt}/{retries}: {e}")

        if attempt < retries:
            time.sleep(retry_delay)

    return ""  # คืน empty string ถ้า retry ทั้งหมดล้มเหลว


def parse_answer(raw_text: str) -> int:
    """แปลง response ของ model เป็นตัวเลข 1-10
    ลำดับการ parse:
    1. หา word boundary '10' หรือ '1'-'9'
    2. หา digit ตัวแรกใน string
    3. fallback → 9 (ไม่มีข้อมูลในฐานข้อมูล)
    """
    if not raw_text:
        return 9  # ถ้า response ว่างเปล่า = ไม่รู้คำตอบ

    # ลอง match ด้วย word boundary ก่อน ('10' ต้องมาก่อน '1' เพื่อกัน conflict)
    match = re.search(r'\b(10|[1-9])\b', raw_text)
    if match:
        return int(match.group(1))

    # Fallback — หา digit อะไรก็ได้ใน string
    digits = re.findall(r'10|[1-9]', raw_text)
    if digits:
        return int(digits[0])

    return 9  # default: ไม่มีข้อมูล


def fallback_answer(question: str, choices: list[str]) -> tuple[int, str]:
    """Fallback สำหรับคำถามที่ retry ทั้งหมดแล้วยังล้มเหลว
    ตรวจสอบว่าคำถามน่าจะ out-of-domain หรือไม่มีข้อมูล
    """
    q_lower = question.lower()
    # คำที่บอกว่า out-of-domain
    ood_signals = ["หุ้น", "ดอลลาร์", "bitcoin", "forex", "ราคาน้ำมัน", "ข่าว", "กีฬา"]
    for sig in ood_signals:
        if sig in q_lower:
            return 10, f"ตรวจพบ signal out-of-domain: '{sig}'"
    # default fallback
    return 9, "ไม่สามารถประมวลผลได้หลังจาก retry ทั้งหมด"


print("✅ LLM inference functions พร้อมใช้งาน")

# ทดสอบ strip_think_tags
test_resp = "<think>\nคิดอยู่...\n</think>\n2"
print(f"   strip_think_tags test: '{strip_think_tags(test_resp)}'")
print(f"   parse_answer('2') = {parse_answer('2')}")
print(f"   parse_answer('คำตอบคือ 10 นะ') = {parse_answer('คำตอบคือ 10 นะ')}")


✅ LLM inference functions พร้อมใช้งาน
   strip_think_tags test: '2'
   parse_answer('2') = 2
   parse_answer('คำตอบคือ 10 นะ') = 10


In [ ]:
def test_api_connection(model_key: str = PRIMARY_MODEL) -> bool:
    """ทดสอบ connectivity กับ API endpoint"""
    print(f"🔌 ทดสอบ: {MODEL_NAMES[model_key]}")
    print(f"   Endpoint: {MODEL_ENDPOINTS[model_key]}")
    try:
        response = call_llm(
            messages=[{"role": "user", "content": "สวัสดี ตอบสั้น ๆ ด้วย"}],
            model_key=model_key, max_tokens=50, temperature=0.3, retries=2,
        )
        if response:
            print(f"   ✅ ตอบกลับ: {response[:80]}")
            return True
        print("   ❌ ได้รับ response ว่างเปล่า")
        return False
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return False


test_api_connection(PRIMARY_MODEL)


🔌 ทดสอบ: OpenThaiGPT-ThaiLLM-8B-instruct-v7.2
   Endpoint: http://thaillm.or.th/api/openthaigpt/v1
   ✅ ตอบกลับ: สวัสดีครับ 有什么可以帮


True

## ขั้นตอนที่ 7: Helper Functions

In [ ]:
def get_question_columns(df: pd.DataFrame) -> tuple[str, list[str]]:
    """auto-detect ชื่อ column คำถาม และ column ตัวเลือกจาก DataFrame"""
    cols = list(df.columns)

    # หา question column
    question_col = None
    for candidate in ["question", "คำถาม", "Question", "q"]:
        if candidate in cols:
            question_col = candidate
            break
    if question_col is None:
        # เดา: column string แรกที่ไม่ใช่ id/answer
        question_col = next((c for c in cols if c.lower() not in ("id","answer") and df[c].dtype == object), cols[1])

    # หา choice columns (รองรับทั้ง choice_1..10 และ choice1..10)
    choice_cols = sorted(
        [c for c in cols if re.match(r'choice[_\s]?\d+', c, re.IGNORECASE)],
        key=lambda x: int(re.search(r'\d+', x).group())
    )

    if not choice_cols:
        # Fallback: column ชื่อตัวเลข หรือ option_N
        choice_cols = sorted(
            [c for c in cols if re.match(r'(option|opt|ans|choice)?[_\s]?\d+$', c, re.IGNORECASE)
             and c not in (question_col, 'id', 'answer')],
            key=lambda x: int(re.search(r'\d+', x).group())
        )

    return question_col, choice_cols


question_col, choice_cols = get_question_columns(questions_df)
print(f"📋 Question column : '{question_col}'")
print(f"📋 Choice columns  : {choice_cols}")


📋 Question column : 'question'
📋 Choice columns  : ['choice_1', 'choice_2', 'choice_3', 'choice_4', 'choice_5', 'choice_6', 'choice_7', 'choice_8', 'choice_9', 'choice_10']


## ขั้นตอนที่ 8: Checkpoint / Resume System

In [ ]:
def load_checkpoint(path: Path = CHECKPOINT_FILE) -> dict:
    """โหลด checkpoint ที่บันทึกไว้ก่อนหน้า (ใช้ resume ได้ทันที)"""
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"📂 โหลด checkpoint: {len(data)} คำถามที่ตอบแล้ว")
        return data
    return {}


def save_checkpoint(results: dict, path: Path = CHECKPOINT_FILE) -> None:
    """บันทึก checkpoint ลงไฟล์ (เรียกหลังทุกคำถาม เพื่อกัน crash)"""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


print("✅ Checkpoint functions พร้อมใช้งาน")


✅ Checkpoint functions พร้อมใช้งาน


## ขั้นตอนที่ 9: Parallel Pipeline (พร้อม Retry & Fallback)

In [ ]:
def process_single_question(q_col, c_cols, row, model_key: str, top_k: int = 6) -> dict:
    """ประมวลผลคำถามเดียว — ใช้กับ ThreadPoolExecutor
    1. ดึง context จาก hybrid retrieval
    2. สร้าง prompt
    3. เรียก LLM
    4. แปลงคำตอบ
    """
    q_id     = int(row["id"])
    question = str(row[q_col])
    choices  = [str(row[c]) for c in c_cols if c in row.index]

    print(f"[Thread] เริ่ม Q{q_id}...")
    try:
        context_chunks = hybrid_retrieve(question, top_k=top_k)
        messages       = build_rag_prompt(question, choices, context_chunks)
        raw            = call_llm(messages, model_key=model_key)
        answer         = parse_answer(raw)

        if not isinstance(answer, int) or answer < 1 or answer > 10:
            raise ValueError(f"คำตอบ '{answer}' ไม่อยู่ในช่วง 1-10")

        print(f"[Thread] ✅ Q{q_id} → {answer}")
        return {
            "id": q_id, "answer": answer, "raw": raw,
            "model": model_key, "context_files": [c["filename"] for c in context_chunks],
            "error": None,
        }
    except Exception as e:
        print(f"[Thread] ❌ Q{q_id} error: {e}")
        return {"id": q_id, "answer": None, "raw": None, "model": model_key, "error": str(e)}


def run_batch(df_rows, model_key: str, max_workers: int, top_k: int = 6) -> tuple[dict, list]:
    """รัน batch ของคำถามแบบ parallel ด้วย ThreadPoolExecutor"""
    q_col, c_cols = get_question_columns(questions_df)
    successes, failures = {}, []

    if not df_rows:
        return successes, failures

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_question, q_col, c_cols, row, model_key, top_k): row
                   for _, row in df_rows}
        for future in as_completed(futures):
            row = futures[future]
            try:
                result = future.result()
            except Exception as exc:
                print(f"[Executor] ❌ Q{row['id']}: {exc}")
                failures.append(row)
                continue

            if result["answer"] is None:
                failures.append(futures[future])
            else:
                successes[result["id"]] = result

    return successes, failures


def run_rag_pipeline(
    df:            pd.DataFrame,
    model_key:     str   = PRIMARY_MODEL,
    top_k:         int   = 6,
    max_workers:   int   = MAX_WORKERS,
    rerun_passes:  int   = RERUN_FAILED,
    force_rerun:   bool  = False,
    ckpt_path:     Path  = CHECKPOINT_FILE,
) -> dict:
    """Pipeline หลัก — รวมทุกอย่างในขั้นตอนเดียว:
    ① parallel batch ② retry ③ fallback ④ checkpoint ⑤ บันทึก submission
    """
    # โหลด checkpoint ถ้ามี
    checkpoint = {} if force_rerun else load_checkpoint(ckpt_path)
    # แยกคำถามที่ตอบแล้ว กับ ที่ยังค้างอยู่
    pending = [(i, row) for i, row in df.iterrows() if str(row["id"]) not in checkpoint]
    print(f"\n🚀 เริ่ม Pipeline: {len(pending)} คำถาม (skip {len(checkpoint)} จาก checkpoint)")
    print(f"   model={MODEL_NAMES[model_key]}, workers={max_workers}, retry={rerun_passes}")
    start_time = time.time()

    answers = dict(checkpoint)  # เริ่มจาก checkpoint

    # ① รันรอบแรกแบบ parallel
    batch_results, failed_rows = run_batch(pending, model_key, max_workers, top_k)
    for q_id, res in batch_results.items():
        answers[str(q_id)] = res
        save_checkpoint(answers, ckpt_path)  # บันทึกทุกคำถาม

    # ② retry คำถามที่ล้มเหลว (แบบ sequential เพื่อความปลอดภัย)
    for pass_idx in range(rerun_passes):
        if not failed_rows:
            break
        print(f"\n🔄 Retry รอบ {pass_idx+1}/{rerun_passes} ({len(failed_rows)} คำถาม)")
        retry_results, failed_rows = run_batch(failed_rows, model_key, max_workers=1, top_k=top_k)
        for q_id, res in retry_results.items():
            answers[str(q_id)] = res
            save_checkpoint(answers, ckpt_path)

    # ③ fallback คำถามที่ยังเหลือ
    if failed_rows:
        print(f"\n🆘 Fallback สำหรับ {len(failed_rows)} คำถามที่ล้มเหลวทั้งหมด")
        q_col, c_cols = get_question_columns(df)
        for _, row in failed_rows:
            q_id     = int(row["id"])
            question = str(row[q_col])
            choices  = [str(row[c]) for c in c_cols if c in row.index]
            ans, reason = fallback_answer(question, choices)
            answers[str(q_id)] = {"id": q_id, "answer": ans, "raw": f"[fallback] {reason}",
                                  "model": "fallback", "context_files": [], "error": None}
            print(f"   [Fallback] Q{q_id} → {ans} ({reason})")
            save_checkpoint(answers, ckpt_path)

    elapsed = time.time() - start_time
    print(f"\n🏁 เสร็จสิ้น {len(answers)}/{len(df)} คำถาม ใน {elapsed:.1f} วินาที")
    return answers


print("✅ Parallel pipeline functions พร้อมใช้งาน")


✅ Parallel pipeline functions พร้อมใช้งาน


## ขั้นตอนที่ 10: Multi-Model Ensemble + Majority Voting

In [ ]:
def run_ensemble(
    df:            pd.DataFrame,
    model_keys:    list[str] = None,
    top_k:         int       = 6,
    max_workers:   int       = MAX_WORKERS,
) -> tuple[dict, dict]:
    """รันหลาย model แล้วใช้ majority voting เลือกคำตอบสุดท้าย
    - แต่ละ model จะมี checkpoint file แยกกัน
    - tie-break: PRIMARY_MODEL ได้รับสิทธิ์ก่อน
    """
    if model_keys is None:
        model_keys = list(MODEL_ENDPOINTS.keys())

    all_results = {}

    for model_key in model_keys:
        print(f"\n{'='*55}")
        print(f"🤖 รัน model: {MODEL_NAMES[model_key]}")
        print(f"{'='*55}")
        ckpt = Path(f"rag_checkpoint_{model_key}.json")  # checkpoint แยกต่างหากต่อ model
        model_results = run_rag_pipeline(
            df=df, model_key=model_key, top_k=top_k,
            max_workers=max_workers, ckpt_path=ckpt,
        )
        all_results[model_key] = model_results

    # Majority voting — ทุก model โหวต แล้วเอา majority
    print("\n🗳️  Majority voting...")
    final_answers = {}
    for _, row in df.iterrows():
        qid   = str(row["id"])
        votes = [
            all_results[mk][qid]["answer"]
            for mk in model_keys
            if qid in all_results.get(mk, {})
        ]
        if votes:
            vote_counter      = Counter(votes)
            final_answers[qid] = vote_counter.most_common(1)[0][0]
        else:
            final_answers[qid] = 9  # fallback ถ้าไม่มี vote เลย

    print(f"✅ Ensemble เสร็จ — {len(final_answers)} คำตอบ")
    return final_answers, all_results
    ensemble_answers, all_model_results = run_ensemble(
    df=questions_df,
    model_keys=["openthaigpt", "pathumma", "typhoon", "kbtg"],
    top_k=6,
)


print("✅ Ensemble function พร้อม — uncomment เพื่อรัน")


✅ Ensemble function พร้อม — uncomment เพื่อรัน


## ขั้นตอนที่ 11: Run Pipeline & Generate Submission

In [ ]:
# รัน ensemble ทั้ง 4 model แล้วใช้ majority voting
# หมายเหตุ: ใช้เวลา ~4x และ quota ~4x ของ single model
ensemble_answers, all_model_results = run_ensemble(
    df=questions_df,
    model_keys=["openthaigpt", "pathumma", "typhoon", "kbtg"],
    top_k=6,
    max_workers=MAX_WORKERS,
)

# สรุปการกระจายคำตอบ
answer_dist = Counter(v for v in ensemble_answers.values())
print("\n📊 การกระจายคำตอบ (Ensemble):")
for ans in sorted(answer_dist):
    label = " ← ไม่มีข้อมูล" if ans == 9 else " ← คำถามไม่เกี่ยวข้อง" if ans == 10 else ""
    print(f"   ตัวเลือก {ans:2d}: {answer_dist[ans]:3d} ครั้ง{label}")


In [ ]:
# สร้าง submission จาก ensemble results
submission_df = generate_submission(
    results=ensemble_answers,   # ใช้ ensemble แทน single model
    df=questions_df,
    output_path=SUBMISSION_CSV,
    is_ensemble=True,           # บอกว่าเป็น ensemble (values เป็น int โดยตรง)
)
display(submission_df.head(10))

# Validate format
assert list(submission_df.columns) == ["id", "answer"], "❌ Columns ต้องเป็น ['id', 'answer']"
assert len(submission_df) == len(questions_df), f"❌ ต้องมี {len(questions_df)} rows"
assert submission_df["answer"].between(1, 10).all(), "❌ คำตอบต้องอยู่ระหว่าง 1-10"
assert not submission_df["id"].duplicated().any(), "❌ พบ ID ซ้ำ!"
print("\n✅ Submission valid!")
print(f"   Answer range: {submission_df['answer'].min()} – {submission_df['answer'].max()}")

## ขั้นตอนที่ 12: Analysis & Debug

In [ ]:
def analyze_results(results: dict, df: pd.DataFrame) -> None:
    """วิเคราะห์ผลลัพธ์: distribution, empty response, top retrieved docs"""
    answered = [v for v in results.values() if isinstance(v, dict)]
    empty    = [v for v in answered if not v.get("raw")]

    print(f"📊 วิเคราะห์ผลลัพธ์")
    print(f"   ตอบแล้ว: {len(answered)}/{len(df)} คำถาม")
    print(f"   Response ว่างเปล่า: {len(empty)} คำถาม")

    answer_counts = Counter(v["answer"] for v in answered)
    print("\n   การกระจายคำตอบ:")
    for i in range(1, 11):
        label = ""
        if i == 9:  label = " ← ไม่มีข้อมูลในฐานข้อมูล"
        if i == 10: label = " ← คำถามไม่เกี่ยวข้อง"
        print(f"   {i:2d}: {answer_counts.get(i, 0):3d} ครั้ง{label}")

    # เอกสารที่ถูก retrieve บ่อยที่สุด
    all_files = [f for v in answered for f in v.get("context_files", [])]
    print("\n   Top 10 เอกสารที่ถูก retrieve บ่อยที่สุด:")
    for fname, cnt in Counter(all_files).most_common(10):
        print(f"   {fname}: {cnt} ครั้ง")


analyze_results(single_model_results, questions_df)


In [ ]:
def debug_question(qid: str, results: dict, df: pd.DataFrame, top_k: int = 6) -> None:
    """แสดงรายละเอียดครบทุกอย่างของคำถามเดียว — ใช้ debug ได้ตรง ๆ"""
    q_col, c_cols = get_question_columns(df)
    row = df[df["id"].astype(str) == str(qid)]
    if row.empty:
        print(f"❌ ไม่พบคำถาม id '{qid}'")
        return

    row      = row.iloc[0]
    question = str(row[q_col])
    choices  = [str(row[c]) for c in c_cols if c in row.index]

    print(f"{'='*70}")
    print(f"🔎 Question ID  : {qid}")
    print(f"   คำถาม       : {question}")
    print("\n   ตัวเลือก:")
    for i, c in enumerate(choices, 1):
        print(f"    {i}. {c}")

    # แสดง context ที่ retrieve ได้
    context = hybrid_retrieve(question, top_k=top_k)
    print(f"\n   Context chunks ({len(context)} รายการ):")
    for i, chunk in enumerate(context, 1):
        print(f"\n   [{i}] {chunk['filename']} (score={chunk['hybrid_score']:.3f})")
        print(f"        {chunk['chunk_text'][:200]}...")

    # แสดงคำตอบที่เก็บไว้
    stored = results.get(str(qid))
    if stored:
        print(f"\n   📥 Raw response : {stored.get('raw', '(ว่าง)')}")
        print(f"   ✅ คำตอบ       : {stored.get('answer')}")
    else:
        print("\n   ⚠️  ยังไม่มีคำตอบสำหรับคำถามนี้")


# ตัวอย่าง: debug คำถามแรก
if questions_df is not None and len(questions_df) > 0:
    sample_qid = str(questions_df.iloc[0]["id"])
    debug_question(sample_qid, single_model_results, questions_df)


In [ ]:
def ask(question: str, model_key: str = PRIMARY_MODEL, top_k: int = 6) -> None:
    """ถามคำถามอิสระ (ไม่ต้องมี choices) — สำหรับ interactive exploration"""
    context = hybrid_retrieve(question, top_k=top_k)
    print(f"🔍 Retrieved {len(context)} chunks:")
    for i, c in enumerate(context, 1):
        print(f"   [{i}] {c['filename']} (score={c['hybrid_score']:.3f})")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"ข้อมูลจากฐานข้อมูล:\n{'\\n\\n'.join(c['chunk_text'] for c in context)}\n\nคำถาม: {question}"},
    ]
    raw = call_llm(messages, model_key=model_key, max_tokens=512, temperature=0.3)
    print(f"\n💬 Response: {raw}")


# ตัวอย่าง:
# ask("ผ้าไหมมัดหมี่ราคาเท่าไร?")
# ask("นโยบายการคืนสินค้าของร้านเป็นอย่างไร?")
print("✅ ask() พร้อมใช้งานแล้ว — ลอง ask('คำถามของคุณ') ได้เลย!")
